# Kazakh (kaz) — Full NLP Pipeline with Cyrillic/Latin Support

Kazakh is the second most resource-rich Turkic language in TurkicNLP. It supports the full pipeline in both Cyrillic and Latin scripts, with Production-quality Apertium FST morphology, Stanza neural parsing (KTB treebank), 25-class NER via KazNERD, and NLLB-200 embeddings/translation.

In [ ]:
# Install TurkicNLP
# pip install turkicnlp          # core (tokenization, transliteration)
# pip install "turkicnlp[stanza]"  # adds POS, lemma, depparse, NER
# pip install "turkicnlp[nllb]"    # adds cross-lingual embeddings + translation
# pip install "turkicnlp[all]"     # all optional dependencies

In [ ]:
import turkicnlp
from turkicnlp import Pipeline

## 1. Download Models

In [ ]:
turkicnlp.download("kaz")

## 2. Script Detection and Cyrillic ↔ Latin Transliteration

In [ ]:
from turkicnlp.scripts import Script
from turkicnlp.scripts.detector import detect_script
from turkicnlp.scripts.transliterator import Transliterator

print("=" * 70)
print("KAZAKH COMPREHENSIVE TRANSLITERATION")
print("=" * 70)
print()

cyrl = "Мен Алматыда тұрамын."
print(f"Original (Cyrillic): {cyrl}")
print()

# Direction 1: Cyrillic → Turkic Common Alphabet (Latin, 2021 official)
print("1. Cyrillic → Turkic Common Alphabet (Latin, 2021 official):")
try:
    t1 = Transliterator("kaz", source=Script.CYRILLIC, target=Script.COMMON_TURKIC)
    common = t1.transliterate(cyrl)
    print(f"   {common}")
except Exception as e:
    print(f"   ⚠ Not supported: {e}")
print()

# Direction 2: Turkic Common → Cyrillic (reverse)
print("2. Turkic Common (Latin) → Cyrillic:")
try:
    t2 = Transliterator("kaz", source=Script.COMMON_TURKIC, target=Script.CYRILLIC)
    back_to_cyrl = t2.transliterate(common if 'common' in locals() else "Men Almatida turamyn.")
    print(f"   {back_to_cyrl}")
    print(f"   ✓ Round-trip match: {cyrl == back_to_cyrl}")
except Exception as e:
    print(f"   ⚠ Not supported: {e}")
print()

# Direction 3: Cyrillic → Latin (explicit)
print("3. Cyrillic → Latin (explicit):")
try:
    t3 = Transliterator("kaz", source=Script.CYRILLIC, target=Script.LATIN)
    latin = t3.transliterate(cyrl)
    print(f"   {latin}")
except Exception as e:
    print(f"   ⚠ Not supported: {e}")
print()

# Direction 4: Latin → Cyrillic
print("4. Latin → Cyrillic:")
try:
    t4 = Transliterator("kaz", source=Script.LATIN, target=Script.CYRILLIC)
    back_to_cyrl_explicit = t4.transliterate(latin if 'latin' in locals() else "Men Almatida turamyn.")
    print(f"   {back_to_cyrl_explicit}")
except Exception as e:
    print(f"   ⚠ Not supported: {e}")
print()

print("=" * 70)
print("Kazakh Scripts Supported:")
print("  • Cyrillic (primary, Soviet legacy)")
print("  • Latin (2021 official transition, COMMON_TURKIC standard)")
print("=" * 70)

## 3. Morphological Analysis (Apertium FST — Production quality)

In [ ]:
nlp_morph = Pipeline(
    "kaz",
    processors=["tokenize", "morph"],
    morph_backend="apertium",
)

# Input in Latin
doc = nlp_morph("Men mektepke baramin.")
print("=== Latin input ===")
for w in doc.words:
    print(f"{w.text:<18} lemma={w.lemma:<12} feats={w.feats}")

In [ ]:
# Input in Cyrillic — pipeline auto-detects and processes transparently
nlp_cyrl = Pipeline(
    "kaz",
    processors=["tokenize", "morph"],
    morph_backend="apertium",
    script="Cyrl",
)
doc = nlp_cyrl("Мен мектепке барамын.")
print("=== Cyrillic input ===")
for w in doc.words:
    print(f"{w.text:<18} lemma={w.lemma:<12} feats={w.feats}")

## 4. POS Tagging, Lemmatisation, and Dependency Parsing (Stanza / KTB)

In [ ]:
nlp_parse = Pipeline(
    "kaz",
    processors=["tokenize", "pos", "lemma", "depparse"],
    script="Cyrl",
)

doc = nlp_parse("Нұр-Сұлтан Қазақстанның астанасы болып табылады.")
print(f"{'Word':<20} {'UPOS':<8} {'Lemma':<20} {'Head':<5} {'Deprel'}")
print("-" * 65)
for w in doc.words:
    print(f"{w.text:<20} {w.upos:<8} {w.lemma:<20} {w.head!s:<5} {w.deprel}")

## 5. Named Entity Recognition — 25 Classes (KazNERD)

In [ ]:
nlp_ner = Pipeline(
    "kaz",
    processors=["tokenize", "pos", "lemma", "ner"],
    script="Cyrl",
)

# "Erzhan went to Astana today."
doc = nlp_ner("Ержан бүгін Астанаға барды.")
for ent in doc.entities:
    print(f"  {ent.text!r:<25} type={ent.type}")

In [ ]:
# Richer sentence with multiple entity types
doc2 = nlp_ner(
    "Қазақстан Республикасының Президенті Қасым-Жомарт Тоқаев "
    "Ақорда сарайында кездесу өткізді."
)
for ent in doc2.entities:
    print(f"  {ent.text!r:<35} type={ent.type}")

## 6. Full Pipeline with CoNLL-U Export

In [ ]:
nlp_full = Pipeline(
    "kaz",
    processors=["tokenize", "pos", "lemma", "depparse", "ner"],
    script="Cyrl",
)

doc = nlp_full("Алматы — Қазақстандағы ең үлкен қала.")

for w in doc.words:
    print(
        f"{w.text:<20} upos={w.upos:<8} "
        f"lemma={w.lemma:<20} ner={w.ner:<8} dep={w.deprel}"
    )
print("\nCoNLL-U:\n", doc.to_conllu())

## 7. Sentence Embeddings and Translation

In [ ]:
import math

turkicnlp.download("kaz", processors=["embeddings", "translate"])

embed = Pipeline("kaz", processors=["embeddings"])
trans = Pipeline("kaz", processors=["translate"], translate_tgt_lang="eng_Latn")

s1 = "Бүгін ауа райы өте жақсы."
s2 = "Ауа-райы бүгін керемет."
d1 = embed(s1)
d2 = embed(s2)

dot = sum(x * y for x, y in zip(d1.embedding, d2.embedding))
norm = (
    math.sqrt(sum(x**2 for x in d1.embedding))
    * math.sqrt(sum(y**2 for y in d2.embedding))
)
print(f"Cosine similarity: {dot/norm:.4f}")
print("Translation:", trans(s1).translation)